# Download Raw QMC Data

The raw (unprocessed) QMC data can be found online at:

- $R = 12.0$ Å: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.13743089.svg)](https://doi.org/10.5281/zenodo.13743089)
- $R = 2.9$ Å: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20217947.svg)](https://doi.org/10.5281/zenodo.20217947)

All processed/reduced data files necessary for producing plots in the paper are included in the repository at `../data`.  However, you can run the code below to download the data for local processing via the `pimcscripts` module.  After decompression this constitutes about `200GB` of data.

Requires installation of the [zenodo_get](https://github.com/dvolgyes/zenodo_get/) package.

In [ ]:
import hg_utils
from hg_utils import Rlab
import dgutils
import pathlib
import numpy as np
import tarfile
import shutil

## Set the value of R

In [ ]:
R = 2.9
dgutils.nb.set_variable("R", R)

In [ ]:
# we have different pinches for different total radii
ΔR = {Rlab(2.9):[0.0,1.0], Rlab(12.0):[0.0,2.0,3.0,4.0]}

## Download the raw data from zenodo

In [ ]:
raw_data_dir = "../data/raw"
pathlib.Path(raw_data_dir+f"/{Rlab(R)}").mkdir(parents=True, exist_ok=True)

In [ ]:
if np.isclose(R,12.0):
    !zenodo_get --doi=10.5281/zenodo.13743089 --output-dir=$raw_data_dir
elif np.isclose(R,2.9):
    !zenodo_get --doi=10.5281/zenodo.20217947 --output-dir=$raw_data_dir
else:
    raise ValueError(r"R = {12.0, 2.9} Å only!")

## Decompress in the `../data/raw` directory then move `gce-position-dR_eq_{ΔR}.npz` files into the correct directories 

In [ ]:
for cΔR in ΔR[Rlab(R)]:
    tar_name = f'{raw_data_dir}/{hg_utils.lab(cΔR)}.tar.gz'
    with tarfile.open(tar_name, "r:gz") as tar:
        tar.extractall(path=raw_data_dir+f"/{Rlab(R)}")

    position_name = pathlib.Path(f'../data/gce-position-{hg_utils.lab(cΔR)}.npz')
    target_dir = f'../data/{Rlab(R)}/{hg_utils.lab(cΔR)}/'
    if position_name.exists():
        shutil.move(position_name, target_dir)